<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_48_telegram_bot/note_lesson_48_telegram_bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Урок 48 — Telegram Bot API

Агрегатор `news_hub` збирає новини, аналізує їх моделлю й захищає запис. Сьогодні в нього з'являється ще один **вхід** — Telegram-бот: `/news` показує останні новини, `/digest` — аналіз LLM, `/subscribe бюджет` — і нова новина про бюджет приходить сама. Бот читає ту саму базу через той самий `NewsRepository`, а його webhook захищений так само, як webhook уроку 47.

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 0 | `news_hub`, база і **двійник Telegram Bot API** прямо в ноутбуці | 0.1 | |
| 1 | **update і Bot API зсередини**: JSON update, об'єкти aiogram, HTTP-виклик вручну, журнал викликів, справжній токен | 1.1–1.5 | |
| 2 | **Bot / Dispatcher / Router**: ехо-бот з нуля, аргументи команд, залежності, один роутер — один диспетчер, команди `news_hub` | 2.1–2.8 | |
| 3 | **порядок handler** у роутері | 3.1–3.2 | |
| 4 | **HTML**: екранування, лапки в посиланні, знайди помилку, ліміт 4096, поділ між рядками | 4.1–4.5 | **Задача 1** |
| 5 | **підписки**: слово підписки, збіг за початком слова, діалог `/subscribe`, ліміт 10 | 5.1–5.4 | **Задача 2** |
| 6 | **сповіщення**: лише нові новини, заблокований чат (403), ліміт (429) | 6.1–6.5 | **Задача 3** |
| 7 | **middleware**: ліміт повідомлень, сесія бази і COMMIT | 7.1–7.2 | |
| 8 | **webhook і polling**: налаштування, секрет у заголовку, адмін-команда, `setWebhook`, `getUpdates` | 8.1–8.6 | |

**Як працювати**

- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (`def`, класи, константи) лише створює їх — **запуск завжди в наступній клітинці**.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- Токен бота й мережа не потрібні: «Telegram» — двійник `tests/telegram_twin.py`, aiohttp-сервер у цьому ноутбуці, що відповідає у форматі Bot API і записує кожен виклик. Лише приклад 1.5 звертається до справжнього Telegram — і тільки якщо задано змінну середовища `BOT_TOKEN`.
- Теорія, покрокові схеми й архітектура — у книзі: [Урок 48](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_48/).

---
## 0. Проєкт, база і двійник Telegram

Клітинка завантажує `news_hub` (у Colab), ставить залежності і створює базу SQLite.

In [ ]:
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_48_telegram_bot"
REPO = "https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026.git"

if not os.path.isdir("news_hub"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

try:
    import aiogram, aiosqlite, bcrypt, fakeredis, fastapi, httpx, jwt  # noqa: F401
    from google import genai  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "news_hub/requirements.txt"], check=True)

os.chdir("news_hub")
sys.path.insert(0, os.getcwd())
os.environ.update({"DATABASE_URL": "sqlite+aiosqlite:///lesson48.db", "REDIS_URL": "fakeredis://", "LLM_PROVIDER": "fake"})
if os.path.exists("lesson48.db"):
    os.remove("lesson48.db")
result = subprocess.run([sys.executable, "-m", "alembic", "upgrade", "head"], capture_output=True, text=True)
print((result.stdout + result.stderr).strip().splitlines()[-1])

### Приклад 0.1. Двійник Telegram, бот і диспетчер

- `TelegramTwin` — aiohttp-сервер на `127.0.0.1` з вільним портом: відповідає як Bot API (`{"ok": true, "result": …}`), відтворює задокументовані правила (неправильний токен → 401, заблокований чат → 403, ліміт → 429, невалідний HTML → 400) і записує кожен виклик.
- `create_bot(token, api_url)` — бот, що ходить на двійник, а не на `api.telegram.org`.
- `create_dispatcher(...)` — диспетчер `news_hub` з усіма командами; адмін — користувач `1001`.
- `say_to(dispatcher, text, ...)` — «користувач надіслав текст»: будує update, проводить його крізь диспетчер і повертає те, що бот надіслав у цей чат.

In [ ]:
import fakeredis
from aiogram import Dispatcher
from aiogram.types import Update

from news_hub.bot.factory import create_bot, create_dispatcher
from news_hub.db import SessionFactory
from news_hub.llm import FakeLLM
from news_hub.models import validate_news
from news_hub.repository import NewsRepository, SubscriptionRepository
from news_hub.snapshot import load_snapshot
from tests.telegram_twin import TelegramTwin, html_error

twin = TelegramTwin()
redis = fakeredis.FakeAsyncRedis(decode_responses=True)


async def say_to(dispatcher: Dispatcher, text: str, chat_id: int = 1001, **user) -> list[str]:
    """Надіслати текст боту від користувача; повертає відповіді бота цьому чату."""
    before = len(twin.sent(chat_id))
    update = Update.model_validate(twin.make_update(text, chat_id=chat_id, **user), context={"bot": bot})
    await dispatcher.feed_update(bot, update)
    return twin.sent(chat_id)[before:]


async def say(text: str, chat_id: int = 1001, **user) -> list[str]:
    """Те саме для диспетчера news_hub."""
    return await say_to(dp, text, chat_id=chat_id, **user)

In [ ]:
await twin.start()
bot = create_bot(twin.token, twin.url)                        # бот ходить на двійник, а не на api.telegram.org
dp = create_dispatcher(SessionFactory, redis, FakeLLM(), admin_ids=frozenset({1001}))
print("двійник:", twin.url, "| бот:", (await bot.get_me()).username)

---
## 1. Update і Bot API зсередини

Telegram надсилає боту **update** — JSON з повідомленням користувача. Бот відповідає HTTP-викликами `https://api.telegram.org/bot<TOKEN>/<метод>`. Шлях одного повідомлення (текстова схема):

```text
Анна в Telegram ──► Telegram ── update (JSON) ──► бот: Dispatcher → handler
                                                          │
Анна бачить відповідь ◄── Telegram ◄── POST /bot<TOKEN>/sendMessage ──┘
```

### Приклад 1.1. Update — це просто JSON

`twin.make_update(...)` будує такий самий update, який надіслав би Telegram. Знайдемо в ньому текст, id чату і тип першої «сутності» повідомлення.

In [ ]:
import json

update = twin.make_update("/news 3", chat_id=2002, first_name="Анна")
text = update["message"]["text"]
chat_id = update["message"]["chat"]["id"]
entity_type = update["message"]["entities"][0]["type"]

print(json.dumps(update, ensure_ascii=False, indent=1)[:500])
print("text:", text, "| chat_id:", chat_id, "| сутність:", entity_type)
assert (text, chat_id, entity_type) == ("/news 3", 2002, "bot_command")

`entities` каже, що перші 5 символів (`/news`) — команда бота. Саме за цим полем фільтр `Command("news")` відрізняє команду від звичайного тексту.

### Приклад 1.2. Той самий update як об'єкт aiogram

aiogram перетворює JSON на модель pydantic: замість `update["message"]["chat"]["id"]` — `update.message.chat.id`, і дата вже `datetime`.

In [ ]:
obj = Update.model_validate(update, context={"bot": bot})
message = obj.message
print("тип події:", obj.event_type, "|", type(message).__name__)
print("текст:", message.text, "| чат:", message.chat.id, message.chat.type, "| від:", message.from_user.first_name)
print("дата:", message.date, "| команда:", message.entities[0].extract_from(message.text))

### Приклад 1.3. Виклик Bot API вручну

Бібліотека — лише зручна обгортка над HTTP. Викличемо методи двійника напряму через `httpx`: `getMe`, `sendMessage` і `getMe` з неправильним токеном.

In [ ]:
import httpx

async with httpx.AsyncClient(base_url=twin.url) as raw:
    me = (await raw.post(f"/bot{twin.token}/getMe")).json()
    sent = (await raw.post(f"/bot{twin.token}/sendMessage", data={"chat_id": 1001, "text": "Привіт з httpx"})).json()
    wrong = await raw.post("/bot000:WRONG/getMe")

print("getMe:      ", me)
print("sendMessage:", sent["ok"], sent["result"]["text"])
print("чужий токен:", wrong.status_code, wrong.json())

Формат той самий, що в справжнього Telegram: `ok: true` і `result` або `ok: false`, `error_code` і `description`. Токен — частина **шляху** (`/bot<TOKEN>/…`), тому його ніколи не пишуть у журнали й не показують у скриншотах.

### Приклад 1.4. Що саме надіслав бот: журнал двійника

Двійник записує кожен виклик. Надішлемо повідомлення через aiogram і подивимось, які параметри він передав.

In [ ]:
await bot.send_message(1001, "<b>Жирний</b> текст")
method, params = twin.calls[-1]
print(method, params)
print("усі методи досі:", [name for name, _ in twin.calls])

`parse_mode=HTML` aiogram додав сам: так налаштовано бота в `create_bot` (`DefaultBotProperties(parse_mode=ParseMode.HTML)`). Тому **кожна** відповідь бота розбирається як HTML — і далі це стане важливим.

### Приклад 1.5. Справжній Telegram — лише з токеном

Якщо в тебе є тестовий бот від @BotFather, задай змінну середовища `BOT_TOKEN` перед запуском ноутбука. Клітинка лише питає `getMe` (нічого не надсилає). Без токена вона пропускає запит і пояснює чому.

In [ ]:
live_token = os.getenv("BOT_TOKEN")
if not live_token:
    print("BOT_TOKEN не задано — справжній Telegram пропущено; решта ноутбука працює з двійником")
else:
    live_bot = create_bot(live_token)                       # без api_url — https://api.telegram.org
    try:
        live_me = await live_bot.get_me()
        print("справжній бот:", live_me.username, live_me.id)
    except Exception as error:                              # немає мережі, неправильний токен …
        print("Telegram недоступний:", type(error).__name__, error)
    finally:
        await live_bot.session.close()

---
## 2. Bot, Dispatcher, Router

Бот з aiogram 3 складається з трьох частин:

- **`Bot`** — HTTP-клієнт Bot API: `send_message`, `set_webhook`, …;
- **`Dispatcher`** — отримує update і веде його крізь middleware до потрібного handler;
- **`Router`** — набір handler з фільтрами (`Command("news")`, `F.text`).

### Приклад 2.1. Ехо-бот з нуля

Найменший бот: команда `/echo <текст>` і відповідь на будь-який інший текст. Handler реєструємо методом `router.message(фільтр)(функція)` — те саме, що декоратор `@router.message(фільтр)`.

In [ ]:
from aiogram import F, Router
from aiogram.filters import Command, CommandObject, CommandStart
from aiogram.types import Message

from news_hub.bot.formatting import esc, link, split_message


async def cmd_echo(message: Message, command: CommandObject) -> None:
    await message.answer(f"🔁 {esc(command.args or '(порожньо)')}")


async def echo_text(message: Message) -> None:
    await message.answer(f"Ти написав: {esc(message.text)}")


def build_echo_router() -> Router:
    router = Router(name="echo")
    router.message(Command("echo"))(cmd_echo)
    router.message(F.text)(echo_text)
    return router

In [ ]:
echo_dp = Dispatcher()
echo_dp.include_router(build_echo_router())
for text in ("/echo Привіт, світе", "/echo", "просто текст"):
    print(f"{text!r:22} →", await say_to(echo_dp, text, chat_id=1101))

### Приклад 2.2. Що є в `CommandObject`

Фільтр `Command` розбирає текст і передає в handler параметр `command`: назву команди, аргументи (усе після пробілу) і префікс. Подивимось на кілька варіантів, зокрема команду з іменем бота (`/echo@news_hub_bot` — так пишуть у групах).

In [ ]:
seen: list[CommandObject] = []


async def remember_command(message: Message, command: CommandObject) -> None:
    seen.append(command)


probe_router = Router()
probe_router.message(Command("echo"))(remember_command)
probe_dp = Dispatcher()
_ = probe_dp.include_router(probe_router)

In [ ]:
for text in ("/echo", "/echo один два", "/echo@news_hub_bot у групі", "/ECHO великими"):
    await say_to(probe_dp, text, chat_id=1102)
for command in seen:
    print(f"command={command.command!r:8} args={command.args!r:16} mention={command.mention!r}")

`args` — `None`, якщо аргументів немає, тому в handler пишуть `command.args or ""`. `/ECHO` не спрацював зовсім: команди чутливі до регістру (`Command("echo", ignore_case=True)` це змінює).

### Приклад 2.3. Залежності в параметрах handler

aiogram «впорскує» в handler значення з `data` **за іменем параметра** — як `Depends` у FastAPI. Middleware кладе туди що завгодно. Handler, якому значення не потрібне, просто не оголошує параметр.

In [ ]:
from aiogram import BaseMiddleware


class GreetingMiddleware(BaseMiddleware):
    """Кладе в data значення, яке handler отримає параметром greeting."""

    async def __call__(self, handler, event, data):
        data["greeting"] = "Добрий вечір"
        return await handler(event, data)


async def cmd_hi(message: Message, greeting: str) -> None:          # greeting — з middleware
    await message.answer(f"{greeting}, {esc(message.from_user.first_name)}!")


async def cmd_plain(message: Message) -> None:                      # greeting не оголошено — і не потрібен
    await message.answer("без залежностей")


di_router = Router()
di_router.message(Command("hi"))(cmd_hi)
di_router.message(Command("plain"))(cmd_plain)
di_dp = Dispatcher()
di_dp.message.middleware(GreetingMiddleware())
_ = di_dp.include_router(di_router)

In [ ]:
print(await say_to(di_dp, "/hi", chat_id=1103, first_name="Остап"))
print(await say_to(di_dp, "/plain", chat_id=1103))

Так `news_hub` передає в handler репозиторії, Redis і LLM: `InjectMiddleware` кладе в `data` ключі `news`, `subscriptions`, `redis`, `llm`, `admin_ids`, а `cmd_news(message, command, news)` бере лише те, що йому треба.

### Приклад 2.4. Один роутер — один диспетчер

У стартовому коді роутер був **змінною модуля**. Що буде, якщо в одному процесі створити два диспетчери (API, тести й ноутбук так і роблять)?

In [ ]:
shared_router = Router(name="shared")
first_dp, second_dp = Dispatcher(), Dispatcher()
first_dp.include_router(shared_router)
try:
    second_dp.include_router(shared_router)
except RuntimeError as error:
    print("RuntimeError:", error)

aiogram прив'язує Router лише до **одного** Dispatcher. Тому в `news_hub` роутер будує функція `handlers.build_router()` — новий на кожен диспетчер, як `build_echo_router()` вище.

### Приклад 2.5. `/start` і `/help` бота `news_hub`

In [ ]:
for reply in await say("/start", chat_id=1104, first_name="Олена") + await say("/help", chat_id=1104):
    print(reply, "\n---")

### Приклад 2.6. Перша відповідь з бази: `/news 3`

Зберігаємо знімок новин у базу (`load_snapshot()` → `validate_news` → `NewsRepository.insert_new`, потім COMMIT) і надсилаємо боту `/news 3`.

In [ ]:
valid, _ = validate_news(load_snapshot())
async with SessionFactory() as session:
    saved = len(await NewsRepository(session).insert_new(valid))
    await session.commit()
[reply] = await say("/news 3", chat_id=2002, first_name="Анна")

print("збережено:", saved)
print(reply)
assert saved > 0 and reply.startswith("📰 <b>Останні новини</b> (3)") and reply.count("<a href=") == 3

Бот не має свого SQL: `cmd_news` викликає той самий `NewsRepository.latest`, що й `GET /api/news`. Заголовки йдуть у посилання через `link(...)`, категорія — через `esc(...)`.

### Приклад 2.7. Аргументи `/news`: межі й помилки

**🔮 Прогноз:** що відповість бот на `/news abc`, `/news 0` і `/news 50`?

In [ ]:
for text in ("/news abc", "/news 0", "/news 50"):
    replies = await say(text, chat_id=1105)
    print(f"{text:10} → {replies[0].splitlines()[0]}")

<details>
<summary>Відповідь</summary>

`/news abc` — підказка «Використання: /news [1–10]». `/news 0` — одна новина, `/news 50` — десять: `max(1, min(N, 10))` обрізає число до меж, а не відмовляє. Межа зверху важлива: інакше одна команда могла б вимагати тисячі рядків з бази.
</details>

### Приклад 2.8. `/digest`: аналіз LLM у боті

`/digest` бере 10 останніх новин і аналізує не більше трьох нових — тим самим `analyze_news`, що й API. LLM у ноутбуці — `FakeLLM`: вона відповідає за сценарієм, без мережі й ключа.

In [ ]:
[digest] = await say("/digest", chat_id=1106)
print(digest[:900])

---
## 3. Порядок handler

Шлях update крізь диспетчер `news_hub` (текстова схема):

```text
«/subscribe Бюджет» від 2002
  outer-middleware: ліміт 1 ≤ 20? ── так ──► handler по черзі: start, help, news, digest — ні
  ──► subscribe? ── так ──► inner-middleware: сесія бази ──► INSERT ──► COMMIT ──► «✅ Підписка…»

«привіт» від 2002
  ліміт 2 ≤ 20? ── так ──► жодна команда не підійшла ──► F.text? ── так ──► «Не знаю такої команди»
```

### Приклад 3.1. У якому порядку роутер перевіряє handler

In [ ]:
order = [handler.callback.__name__ for handler in dp.sub_routers[0].message.handlers]
print(order)
assert order[0] == "cmd_start" and order[-1] == "unknown" and "cmd_subscribe" in order

**🔮 Прогноз:** що відповів би бот на `/help`, якби `unknown` (фільтр `F.text`) стояв першим?

### Приклад 3.2. «Будь-який текст» першим

Зберемо роутер з тих самих handler `news_hub`, але `unknown` зареєструємо першим. Диспетчер — без middleware, тож handler, яким потрібна база, тут не викликаємо.

In [ ]:
from news_hub.bot import handlers


def build_wrong_router() -> Router:
    router = Router(name="wrong-order")
    router.message(F.text)(handlers.unknown)               # помилка: «будь-який текст» першим
    router.message(CommandStart())(handlers.cmd_start)
    router.message(Command("help"))(handlers.cmd_help)
    return router


wrong_dp = Dispatcher()
_ = wrong_dp.include_router(build_wrong_router())

In [ ]:
for text in ("/help", "/start", "привіт"):
    print(f"{text:7} →", await say_to(wrong_dp, text, chat_id=1107))

<details>
<summary>Відповідь</summary>

«Не знаю такої команди» — на `/help`, `/start` і на будь-яку іншу команду. aiogram бере **перший** handler, чий фільтр підійшов, а `F.text` підходить до будь-якого тексту, зокрема до команд. Тому «будь-який текст» — останнім, а порядок решти команд не важливий: їхні фільтри не перетинаються.
</details>

---
## 4. HTML: екранування і ліміт 4096

Бот відповідає з `parse_mode=HTML`: `<b>`, `<i>`, `<a href>`. Telegram (і двійник) відхиляє все повідомлення, якщо «<», «>» чи «&» стоять поза тегом. `html_error(text)` з двійника каже, що не так, або повертає `None`.

### Приклад 4.1. Рядок як є і після `esc`

Для кожного рядка з `SAMPLES` перевіряємо `<b>рядок</b>` до і після `esc` (`html.escape(text, quote=False)`).

In [ ]:
SAMPLES = ["Олена", "<Олена>", "Tom & Jerry", "ціна < 100 грн", "&amp; уже екрановано", "5 > 3"]

In [ ]:
for sample in SAMPLES:
    raw, safe = html_error(f"<b>{sample}</b>"), html_error(f"<b>{esc(sample)}</b>")
    print(f"{sample!r:24} як є: {'ok' if raw is None else raw[:60]:62} після esc: {esc(sample)!r} {'ok' if safe is None else safe}")

Ламають розмітку «<», «&» і «>». `&amp;` уже є правильною сутністю, тож «як є» він проходить — а після `esc` перетворюється на `&amp;amp;`: користувач побачить `&amp;`. **Правило:** екранувати **один раз**, саме в момент вставки в HTML, і лише те, що прийшло ззовні.

### Приклад 4.2. Посилання: лапки в адресі

`esc` не чіпає лапки — поза тегом вони не потрібні. А всередині значення атрибута `href` лапка **закриває** атрибут. Порівняємо наївне посилання з `link(url, title)` і подивимось, яку адресу насправді прочитає HTML-парсер.

In [ ]:
from html.parser import HTMLParser


class HrefReader(HTMLParser):
    """Дістає значення href з першого <a> — так, як його прочитає HTML-парсер."""

    def handle_starttag(self, tag, attrs):
        if tag == "a" and not hasattr(self, "href"):
            self.href = dict(attrs).get("href")


def read_href(markup: str) -> str | None:
    reader = HrefReader()
    reader.feed(markup)
    return getattr(reader, "href", None)


URL = 'https://www.rbc.ua/search?q="бюджет"&page=2'
naive = f'<a href="{URL}">{esc("Бюджет & податки")}</a>'
safe_link = link(URL, "Бюджет & податки")

In [ ]:
print("наївно:", naive)
print("  парсер бачить href =", read_href(naive))
print("link():", safe_link)
print("  парсер бачить href =", read_href(safe_link))
print("двійник: наївно", html_error(naive), "| link()", html_error(safe_link))

У наївному посиланні адреса обірвалась на першій лапці: `https://www.rbc.ua/search?q=`. Двійник цього не помітив — він перевіряє теги й сутності, а не вміст атрибутів (і це чесна межа двійника). `link` екранує адресу з `quote=True`, і парсер читає її повністю.

### Приклад 4.3. Знайди помилку: ім'я в привітанні

`old_start` — handler `/start` зі стартового коду. Його тест з ім'ям «Олена» зелений. Зберемо диспетчер з одним цим handler і надішлемо `/start` від імені `Tom & Jerry`.

In [ ]:
from aiogram.exceptions import TelegramBadRequest


async def old_start(message: Message) -> None:
    await message.answer(f"Привіт, <b>{message.from_user.first_name}</b>! 🤖")        # вразливо: ім'я без esc


async def fixed_start(message: Message) -> None:
    await message.answer(f"Привіт, <b>{esc(message.from_user.first_name)}</b>! 🤖")


def dispatcher_with(command_filter, handler) -> Dispatcher:
    """Диспетчер з одним handler — для коротких прикладів."""
    router = Router()
    router.message(command_filter)(handler)
    new_dp = Dispatcher()
    new_dp.include_router(router)
    return new_dp

In [ ]:
for name in ("Олена", "<Олена>", "Tom & Jerry"):
    try:
        replies = await say_to(dispatcher_with(CommandStart(), old_start), "/start", chat_id=5005, first_name=name)
        print(f"старий, {name!r:14}: надіслано {replies}")
    except TelegramBadRequest as error:
        print(f"старий, {name!r:14}: {type(error).__name__}: {error}")
print("новий:", await say_to(dispatcher_with(CommandStart(), fixed_start), "/start", chat_id=5005, first_name="Tom & Jerry"))

<details>
<summary>Чому так</summary>

Ім'я вставлено в HTML без екранування: «<» чи «&» поза тегом — зламана розмітка, і Telegram відхиляє **все** повідомлення (`can't parse entities`). Бот мовчить, а користувач не розуміє чому. Тест перевіряв лише «звичайне» ім'я. Виправлення — `esc(...)` для всього, що прийшло ззовні.
</details>

### Приклад 4.4. Ліміт 4096 символів

Telegram приймає до 4096 символів в одному повідомленні. Перевіримо межу на двійнику.

In [ ]:
for size in (4096, 4097):
    try:
        await bot.send_message(1108, "я" * size)
        print(size, "символів: надіслано")
    except TelegramBadRequest as error:
        print(size, "символів:", error)

### Приклад 4.5. Довге повідомлення: різати між рядками

`lines` — 300 рядків-посилань. Порівнюємо два способи поділу: кожні 4000 символів (як `split_long_message` стартового коду) і `split_message(lines)` — лише між рядками.

In [ ]:
lines = [link(f"https://www.rbc.ua/ukr/news/{i}.html", f"Новина {i}: бюджет & податки") for i in range(300)]
text = "\n".join(lines)
chunks = [text[i: i + 4000] for i in range(0, len(text), 4000)]
parts = split_message(lines)

In [ ]:
bad_chunks = [html_error(chunk) for chunk in chunks if html_error(chunk) is not None]
bad_parts = sum(html_error(part) is not None for part in parts)
print("кожні 4000:", len(chunks), "частин, невалідних", len(bad_chunks), "| наприклад:", bad_chunks[0][:70])
print("між рядками:", len(parts), "частин, невалідних", bad_parts, "| довжини:", [len(p) for p in parts])
assert bad_chunks and bad_parts == 0 and all(len(p) <= 4096 for p in parts)

Розріз кожні 4000 символів падає посеред `<a href=…>` чи `&amp;` — і кожна частина стає невалідною. Бот складає повідомлення з **рядків**, кожен рядок — цілий HTML, тож ділимо лише між ними.

### Задача 1. Команда `/hello`

Допиши handler `cmd_hello(message, command)`:

- `/hello <ім'я>` → `Привіт, <b>ім'я</b>!` — ім'я **екрановане** (`esc`), зайві пробіли навколо прибрано;
- `/hello` без імені → `Використання: /hello &lt;ім'я&gt;`.

Перевірка зареєструє його в диспетчері з фільтром `Command("hello")` і надішле три команди.

In [ ]:
async def cmd_hello(message: Message, command: CommandObject) -> None:
    """/hello <ім'я> → привітання з екранованим ім'ям; без імені — підказка."""
    # BEGIN SOLUTION
    name = (command.args or "").strip()
    if not name:
        await message.answer("Використання: /hello &lt;ім'я&gt;")
        return
    await message.answer(f"Привіт, <b>{esc(name)}</b>!")
    # END SOLUTION

In [ ]:
hello_dp = dispatcher_with(Command("hello"), cmd_hello)
answers = {}
for text in ("/hello  Остап ", "/hello Tom & Jerry", "/hello"):
    try:
        answers[text] = await say_to(hello_dp, text, chat_id=6006)
    except TelegramBadRequest as error:
        answers[text] = f"Telegram відхилив: {error}"
print(answers)
assert answers["/hello  Остап "], "бот нічого не відповів: handler має викликати message.answer(...)"
assert answers["/hello  Остап "] == ["Привіт, <b>Остап</b>!"], "прибери пробіли навколо імені й візьми формат «Привіт, <b>ім'я</b>!»"
assert answers["/hello Tom & Jerry"] == ["Привіт, <b>Tom &amp; Jerry</b>!"], "ім'я треба екранувати: esc(name)"
assert answers["/hello"] == ["Використання: /hello &lt;ім'я&gt;"], "без імені — підказка (кутові дужки теж екрановані)"
print("✅ Задача 1 розв'язана")

---
## 5. Підписки

`/subscribe <слово>` записує пару «чат + слово» в таблицю `subscriptions`. Після збору новин чат отримує ті, де є його слово.

### Приклад 5.1. Яке слово можна підписати

`normalize_keyword` приймає одне слово 2–40 символів (літери, цифри, апостроф, дефіс) і зводить регістр (`casefold`); інакше — `None`.

In [ ]:
from news_hub.bot.formatting import matches, normalize_keyword

RAW_KEYWORDS = ["Бюджет", "  ЗСУ ", "п'ятниця", "Кривий-Ріг", "я", "два слова", "<script>", "x" * 41]

In [ ]:
for raw in RAW_KEYWORDS:
    print(f"{raw[:20]!r:24} → {normalize_keyword(raw)!r}")

### Приклад 5.2. Збіг за початком слова

Українська змінює закінчення, тож точний збіг слова пропускав би більшість новин, а пошук підрядка будь-де давав би хибні збіги. `matches(keyword, title)` — чи **починається** якесь слово заголовка з ключового.

In [ ]:
CASES = [("бюджет", "Бюджету бракує 100 млрд"), ("рада", "Верховна Рада ухвалила закон"),
         ("рада", "Це зрада, кажуть експерти"), ("газ", "Газета вийшла вранці"), ("погода", "Прогноз погоди на тиждень")]

In [ ]:
for keyword, title in CASES:
    print(f"{keyword:7} | {title:32} | підрядок: {keyword in title.casefold()!s:5} | початок слова: {matches(keyword, title)}")

«Зрада» не збігається з «рада» — добре. «Газета» збігається з «газ» — це межа методу: початок слова не є коренем. А «погоди» не збігається з «погода» — інше закінчення вже в основі; підписуватись краще на коротшу основу («погод»).

### Задача 2. Які підписки спрацювали

Допиши `matched_keywords(title, keywords)`: список тих ключових слів з `keywords` (у тому ж порядку), з яких **починається** хоча б одне слово заголовка. Слова заголовка — `re.findall(r"[\w'’-]+", title.casefold())`; ключові слова вже в нижньому регістрі.

In [ ]:
import re


def matched_keywords(title: str, keywords: list[str]) -> list[str]:
    """Ключові слова, з яких починається хоча б одне слово заголовка (порядок — як у keywords)."""
    # BEGIN SOLUTION
    words = re.findall(r"[\w'’-]+", title.casefold())
    return [keyword for keyword in keywords if any(word.startswith(keyword) for word in words)]
    # END SOLUTION

In [ ]:
KEYWORDS = ["бюджет", "рада", "погод", "газ"]
checks = {
    "Верховна Рада ухвалила бюджет-2027": ["бюджет", "рада"],
    "Це зрада, кажуть експерти": [],
    "ПОГОДА: газети пишуть про бюджету дефіцит": ["бюджет", "погод", "газ"],
    "": [],
}
got = {title: matched_keywords(title, KEYWORDS) for title in checks}
for title, found in got.items():
    print(f"{title!r:46} → {found}")
assert all(isinstance(found, list) for found in got.values()), "функція має повертати список"
assert got["Це зрада, кажуть експерти"] == [], "«рада» всередині «зрада» — не збіг: перевіряй початок слова (startswith), а не підрядок"
assert got == checks
assert all(matched_keywords(title, [k]) == ([k] if matches(k, title) else []) for title, _ in CASES for k in KEYWORDS), \
    "результат має збігатися з matches() з проєкту"
print("✅ Задача 2 розв'язана")

### Приклад 5.3. Діалог підписки

Анна підписується, дивиться свої підписки, повторює ту саму підписку, скасовує і пробує зайве.

In [ ]:
for text in ("/subscribe Бюджет", "/subscribe бюджет", "/subscribe зеленськ", "/subscriptions",
             "/unsubscribe бюджет", "/unsubscribe футбол", "/subscriptions", "/subscribe два слова"):
    print(f"{text:22} → {(await say(text, chat_id=2002, first_name='Анна'))[0]}")

Повторна підписка нічого не додає (`INSERT … ON CONFLICT DO NOTHING` — пара «чат + слово» унікальна), а слово зводиться до нижнього регістру ще до запису.

### Приклад 5.4. Не більше 10 підписок

Без ліміту один чат міг би підписатися на тисячі слів, і кожен збір перебирав би їх усі.

In [ ]:
words = ["бюджет", "рада", "погод", "газ", "ціна", "уряд", "нбу", "долар", "євро", "війна", "мир"]
replies = [(await say(f"/subscribe {word}", chat_id=9009))[0] for word in words]
print("перші:", replies[:2])
print("десята:", replies[9])
print("одинадцята:", replies[10])

Прибираємо підписки цього чату тим самим методом, яким розсилка прибирає чати, що заблокували бота, — щоб вони не втручались у приклади розділу 6.

In [ ]:
async with SessionFactory() as session:
    removed = await SubscriptionRepository(session).remove_chat(9009)
    await session.commit()
print("видалено підписок чату 9009:", removed)

---
## 6. Сповіщення

Після збору `notify_subscribers(bot, subscriptions, нові_новини)` надсилає кожному чату **одне** повідомлення з усіма збігами. Що відбувається за одного збору (текстова схема):

```text
збір: validate_news → insert_new (RETURNING лише нових) → COMMIT
підписки by_chat():   1001: зеленськ | 2002: погода | 3003: зеленськ, україн
для кожного чату:     є збіги? ── ні ──► нічого
                               └─ так ─► sendMessage ── 200 ──► sent += 1
                                                     ├─ 403 ──► remove_chat (бота заблокували)
                                                     └─ 429 ──► чекати retry_after і ще раз
повторний збір тих самих новин: insert_new → [] → розсилки немає
```

### Приклад 6.1. Повідомлення для одного чату

`build_messages(keywords, news)` — чисте перетворення «слова + нові новини → частини повідомлення», без Telegram. Візьмемо кілька новин зі знімка.

In [ ]:
from news_hub.notify import build_messages, notify_subscribers
from news_hub.repository import NewNews

sample_news = [NewNews(id=i, title=item.title, url=str(item.url)) for i, item in enumerate(valid[:40])]

In [ ]:
messages = build_messages(["зеленськ", "україн"], sample_news)
print(len(messages), "частина;", messages[0][:400] if messages else "збігів немає")
print("без збігів:", build_messages(["футбол"], sample_news))

### Приклад 6.2. Розсилка після нового збору

Анна (`2002`) уже підписана на «зеленськ» (приклад 5.3), Боб (`3003`) підписується на «погода». Щоб отримати **нові** новини, очищаємо базу (`NewsRepository.clear`) і знову робимо `insert_new`.

In [ ]:
await say("/subscribe погода", chat_id=3003, first_name="Боб")

async with SessionFactory() as session:
    news = NewsRepository(session)
    await news.clear()
    new = await news.insert_new(valid)
    report = await notify_subscribers(bot, SubscriptionRepository(session), new)
    await session.commit()
anna, bob = twin.sent(2002)[-1], twin.sent(3003)[-1]

print(report)
print(anna[:300])
print("Боб отримав останнім:", bob[:60])
assert report.chats == 1 and anna.startswith("🔔 Нові новини за підписками (зеленськ):")
assert not bob.startswith("🔔")

Повідомлення отримав лише один чат: у знімку немає новин про погоду, тож Боб бачить останнім своє «✅ Підписка на «погода»».

### Приклад 6.3. Ті самі новини вдруге — нікого не сповіщає

**🔮 Прогноз:** що поверне `insert_new`, якщо зібрати той самий знімок ще раз, без очищення бази?

In [ ]:
async with SessionFactory() as session:
    again = await NewsRepository(session).insert_new(valid)
    second_report = await notify_subscribers(bot, SubscriptionRepository(session), again)
    await session.commit()
print("нових:", len(again), "|", second_report)

<details>
<summary>Відповідь</summary>

Порожній список: `insert_new` повертає (`RETURNING`) лише рядки, які справді вставились, а дублікати за URL пропускаються. Тому повторний збір нікого не турбує вдруге — а стара версія розсилки щогодини надсилала все знову.
</details>

### Приклад 6.4. Бота заблокували

Анна заблокувала бота: `twin.blocked.add(2002)`. Повторюємо збір і розсилку з прикладу 6.2.

In [ ]:
twin.blocked.add(2002)

async with SessionFactory() as session:
    news = NewsRepository(session)
    await news.clear()
    new = await news.insert_new(valid)
    subscriptions = SubscriptionRepository(session)
    report = await notify_subscribers(bot, subscriptions, new)
    await session.commit()
    left = await subscriptions.for_chat(2002)

print(report, "| підписки Анни:", left)
assert report.blocked == [2002] and left == []

Telegram відповідає `403 Forbidden: bot was blocked by the user`. Розсилка видаляє підписки цього чату: інакше кожен збір знову стукав би в чат, який ніколи не відповість. Решта розсилки триває.

### Приклад 6.5. Ліміт Telegram: 429 і `retry_after`

Двійник уміє відповісти `429 Too Many Requests` з `retry_after: 1` на наступний виклик. Розсилка має почекати й спробувати ще раз. Боб підписується на «зеленськ», щоб отримати повідомлення.

In [ ]:
import time

await say("/subscribe зеленськ", chat_id=3003)
twin.retry_after = 1                                  # наступний sendMessage → 429, retry_after=1

async with SessionFactory() as session:
    news = NewsRepository(session)
    await news.clear()
    new = await news.insert_new(valid)
    start = time.perf_counter()
    report = await notify_subscribers(bot, SubscriptionRepository(session), new)
    await session.commit()
elapsed = time.perf_counter() - start
attempts = [params.get("error", "200 OK") for method, params in twin.calls[-2:] if method == "sendMessage"]
print(report, f"| {elapsed:.1f} с | спроби: {attempts}")

Перша спроба — `429`, далі пауза 1 с (`TelegramRetryAfter.retry_after`) і друга — успішна. Справжній Telegram дозволяє боту близько 30 повідомлень за секунду, тому `notify` ще й робить коротку паузу між повідомленнями (`SEND_PAUSE`).

### Задача 3. Своя розсилка з обробкою 403

Допиши `broadcast(bot, chat_ids, text)`: надіслати `text` кожному чату по черзі. Якщо чат заблокував бота (`TelegramForbiddenError`), запам'ятати його й **продовжити**. Повернути словник `{"sent": [...], "blocked": [...]}` зі списками id чатів.

In [ ]:
from aiogram import Bot
from aiogram.exceptions import TelegramForbiddenError


async def broadcast(bot: Bot, chat_ids: list[int], text: str) -> dict[str, list[int]]:
    """Надіслати text усім chat_ids; заблоковані — у "blocked", розсилка триває."""
    # BEGIN SOLUTION
    result: dict[str, list[int]] = {"sent": [], "blocked": []}
    for chat_id in chat_ids:
        try:
            await bot.send_message(chat_id, text)
            result["sent"].append(chat_id)
        except TelegramForbiddenError:
            result["blocked"].append(chat_id)
    return result
    # END SOLUTION

In [ ]:
twin.blocked.add(8008)
announcement = "📢 Завтра о 03:00 — оновлення агрегатора, бот на 5 хвилин замовкне."
outcome = await broadcast(bot, [7001, 8008, 7002], announcement)
print(outcome)
assert isinstance(outcome, dict), "broadcast має повертати словник {'sent': [...], 'blocked': [...]}"
assert outcome == {"sent": [7001, 7002], "blocked": [8008]}, "заблокований чат — у blocked, решта — у sent, порядок — як у chat_ids"
assert twin.sent(7002)[-1] == announcement, "після заблокованого чату розсилка має тривати"
print("✅ Задача 3 розв'язана")

---
## 7. Middleware: ліміт і сесія бази

Диспетчер `news_hub` має два шари:

```text
update ──► [outer: RateLimitMiddleware] ──► роутер шукає handler ──► [inner: InjectMiddleware] ──► handler
             ліміт 20 повідомлень/хв           фільтри Command, F.text       сесія бази, COMMIT після handler
```

### Приклад 7.1. Ліміт повідомлень від одного користувача

Користувач `7007` надсилає 23 повідомлення поспіль.

In [ ]:
counts = []
for i in range(23):
    counts.append(len(await say("/help", chat_id=7007)))
print("відповідей на кожне:", counts)
print("остання відповідь бота:", twin.sent(7007)[-1])
print("ключ у Redis:", await redis.get("rate:bot:tg7007"), "| живе ще", await redis.ttl("rate:bot:tg7007"), "с")

Перші 20 — відповіді, 21-ше — одне попередження «⏳ Забагато запитів», далі бот **мовчить**: відповідати на кожне зайве повідомлення означало б самому порушувати ліміти Telegram. Це той самий `RateLimiter` (`INCR` + `EXPIRE NX`), що й ліміт API в уроці 40.

### Приклад 7.2. Без COMMIT підписка «зникає»

`InjectMiddleware` відкриває сесію бази на кожен update і робить COMMIT після handler. Що буде, якщо COMMIT забути? Зберемо диспетчер з такою «зламаною» версією middleware і тими самими handler `news_hub`.

In [ ]:
from news_hub.bot.middlewares import InjectMiddleware


class InjectWithoutCommit(InjectMiddleware):
    """Вразливо: сесія закривається без COMMIT — усе, що зробив handler, відкочується."""

    async def __call__(self, handler, event, data):
        async with self._factory() as session:
            data["news"] = NewsRepository(session)
            data["subscriptions"] = SubscriptionRepository(session)
            data["redis"], data["llm"], data["admin_ids"] = self._redis, None, self._admin_ids
            return await handler(event, data)                  # немає await session.commit()


broken_dp = Dispatcher()
broken_dp.message.middleware(InjectWithoutCommit(SessionFactory, redis, None, frozenset()))
_ = broken_dp.include_router(handlers.build_router())

In [ ]:
print(await say_to(broken_dp, "/subscribe бюджет", chat_id=7101))
print(await say_to(broken_dp, "/subscriptions", chat_id=7101))
print("а з правильним middleware:", await say("/subscribe бюджет", chat_id=7101), await say("/subscriptions", chat_id=7101))

Бот «пообіцяв» підписку, але INSERT виконався в сесії, яка закрилась із ROLLBACK, — у базі нічого немає. Це як `get_db` у FastAPI: сесія на запит і COMMIT після обробника. Тест `test_subscriptions` у проєкті ловить саме це.

---
## 8. Webhook і polling

Telegram доставляє update одним із двох способів:

```text
є публічна адреса з https? ── ні ──► polling: бот сам питає getUpdates (python -m news_hub.bot)
          │
          так ──► webhook: Telegram сам надсилає POST /api/telegram/webhook
                           із секретом у заголовку X-Telegram-Bot-Api-Secret-Token
Одночасно — ні: поки webhook увімкнено, getUpdates відповідає 409.
```

### Приклад 8.1. Налаштування бота зі змінних середовища

`load_bot_settings(env)`, як і налаштування адміна в уроці 47, або повертає налаштування, або не дає застосунку стартувати з поясненням.

In [ ]:
from news_hub.bot.settings import BotSettings, load_bot_settings

GOOD_SECRET = "notebook-telegram-secret-" + "x" * 32
BOT_ENVS = {
    "без токена": {},
    "polling": {"BOT_TOKEN": "123:ABC", "BOT_ADMIN_IDS": "1001, 2002"},
    "webhook на http": {"BOT_TOKEN": "123:ABC", "TELEGRAM_WEBHOOK_URL": "http://news.example/hook",
                        "TELEGRAM_WEBHOOK_SECRET": GOOD_SECRET},
    "короткий секрет": {"BOT_TOKEN": "123:ABC", "TELEGRAM_WEBHOOK_URL": "https://news.example/hook",
                        "TELEGRAM_WEBHOOK_SECRET": "short"},
    "webhook": {"BOT_TOKEN": "123:ABC", "TELEGRAM_WEBHOOK_URL": "https://news.example/hook",
                "TELEGRAM_WEBHOOK_SECRET": GOOD_SECRET},
}

In [ ]:
for name, env in BOT_ENVS.items():
    try:
        settings = load_bot_settings(env)
        verdict = "бот вимкнено, API працює як раніше" if settings is None else \
            f"webhook={settings.webhook_url}, адміни={sorted(settings.admin_ids)}"
    except RuntimeError as error:
        verdict = f"не стартує: {error}"
    print(f"{name:16} {verdict}")

`https://` обов'язковий: Telegram не надсилає webhook на http. Секрет — 32–256 символів `A-Z a-z 0-9 _ -`: це правило самого Telegram для `secret_token`, і краще дізнатися про нього при старті, ніж від мовчазного бота.

### Приклад 8.2. Webhook у FastAPI: секрет у заголовку

У webhook-режимі Telegram надсилає update на `POST /api/telegram/webhook` з заголовком `X-Telegram-Bot-Api-Secret-Token` — тим самим секретом, який застосунок передав у `setWebhook`. Підключаємо до застосунку нашого бота й диспетчер і ходимо через `httpx.AsyncClient` з `ASGITransport` (застосунок у пам'яті, без порту).

In [ ]:
from news_hub.api import app

SECRET = GOOD_SECRET
HEADER = "X-Telegram-Bot-Api-Secret-Token"
app.state.bot, app.state.dispatcher = bot, dp
app.state.telegram = BotSettings(token=twin.token, api_url=twin.url, webhook_url="https://news.example/hook",
                                 webhook_secret=SECRET)
api = httpx.AsyncClient(transport=httpx.ASGITransport(app=app), base_url="http://test")

In [ ]:
without = await api.post("/api/telegram/webhook", json=twin.make_update("/help", chat_id=4004))
wrong = await api.post("/api/telegram/webhook", json=twin.make_update("/help", chat_id=4004), headers={HEADER: "guess"})
not_update = await api.post("/api/telegram/webhook", json={"hello": "world"}, headers={HEADER: SECRET})
response = await api.post("/api/telegram/webhook", json=twin.make_update("/help", chat_id=4004), headers={HEADER: SECRET})
for name, r in (("без секрету", without), ("чужий секрет", wrong), ("не update", not_update), ("з секретом", response)):
    print(f"{name:13} {r.status_code} {r.json()}")
answer = twin.sent(4004)[-1]
print(answer[:80])
assert (without.status_code, response.status_code) == (401, 200) and answer.startswith("<b>Команди</b>")

Відповідь `200 {"ok": true}` іде **одразу**, а обробка update — у фоні (`BackgroundTasks`): Telegram чекає відповіді недовго і, не дочекавшись, надсилає update повторно. Секрет перевіряє `verify_secret_token` з уроку 47 — порівняння за сталий час. Сміття замість update — `400`, ще до диспетчера.

### Приклад 8.3. Бот вимкнено → 503

Без `BOT_TOKEN` бота немає, а webhook чесно каже, що приймати нікому.

In [ ]:
saved_bot = app.state.bot
app.state.bot = None
try:
    off = await api.post("/api/telegram/webhook", json=twin.make_update("/help"), headers={HEADER: SECRET})
    print(off.status_code, off.json())
finally:
    app.state.bot = saved_bot

### Приклад 8.4. Адмін-команда `/scrape`

`/scrape snapshot` у боті збирає новини — але лише для `admin_ids` (тут `1001`). Надсилаємо її від `4004` і від `1001`.

In [ ]:
[denied] = await say("/scrape snapshot", chat_id=4004)
[allowed] = await say("/scrape snapshot", chat_id=1001)
print("4004:", denied)
print("1001:", allowed)
assert denied == "Ця команда — лише для адміністратора." and allowed.startswith("Зібрано: 168")

Зібрано 168, нових 0: ці новини вже в базі після прикладу 6.5, тож і сповіщень немає.

**🔮 Прогноз:** чому захист у `/scrape` — за `user_id`, а не за `chat_id`?

<details>
<summary>Відповідь</summary>

`chat_id` — розмова, `user_id` — людина. У групі один `chat_id` на всіх учасників: перевірка за чатом дала б право адміна кожному в групі, де є адмін.
</details>

### Приклад 8.5. `setWebhook`, `getWebhookInfo` і конфлікт з polling

У проєкті `lifespan` сам викликає `setWebhook`. Зробимо це вручну на двійнику і спробуємо водночас запитати `getUpdates`.

In [ ]:
from aiogram.exceptions import TelegramConflictError

await bot.set_webhook("https://news.example/api/telegram/webhook", secret_token=SECRET)
info = await bot.get_webhook_info()
print("webhook:", info.url, "| секрет у двійника збігся:", twin.webhook_secret == SECRET)
try:
    await bot.get_updates(timeout=0)
except TelegramConflictError as error:
    print("getUpdates:", error)
await bot.delete_webhook()
print("після deleteWebhook:", repr((await bot.get_webhook_info()).url))

Telegram не віддає update обома способами одночасно. Тому `python -m news_hub.bot` (polling) спершу видаляє webhook. А застосунок у webhook-режимі, зупиняючись, webhook **не** видаляє: під час перезапуску новий процес уже поставив його, і старий прибрав би його в нового.

### Приклад 8.6. Polling вручну

Polling — це цикл: «дай нові update» → обробити → підтвердити (`offset = останній update_id + 1`). `dp.start_polling(bot)` робить це безкінечно; зробимо один оберт руками. `twin.push(...)` кладе update в чергу, як це зробив би Telegram, коли користувач пише боту.

In [ ]:
for text in ("/news 1", "привіт"):
    twin.push(text, chat_id=1201, first_name="Ірина")         # Ірина написала боту двічі
print("у черзі getUpdates:", len(twin.updates))

In [ ]:
updates = await bot.get_updates(offset=0, timeout=0)
print("отримано update:", [u.update_id for u in updates])
for item in updates:
    await dp.feed_update(bot, item)
confirmed = await bot.get_updates(offset=updates[-1].update_id + 1, timeout=0)
print("після підтвердження в черзі:", len(confirmed))
for reply in twin.sent(1201):
    print("→", reply.splitlines()[0])

Той самий `dp` обробляє update і з polling, і з webhook: різниться лише **спосіб доставки**. Тому в `news_hub` одні й ті самі фабрики `create_bot` / `create_dispatcher` збирають бота для webhook у FastAPI, для `python -m news_hub.bot` і для тестів.

Прибираємо за собою: клієнт API, сесію бота, двійник і стан застосунку.

In [ ]:
await api.aclose()
await bot.session.close()
await twin.stop()
app.state.bot = app.state.dispatcher = app.state.telegram = None
print("готово")

---
## Самоперевірка

1. Що таке update і хто його надсилає в режимах polling і webhook?
2. Як handler отримує `news` чи `command`, якщо ніхто не викликає його з цими аргументами?
3. Чому `F.text` — останній handler, а роутер будує функція, а не змінна модуля?
4. Що станеться з повідомленням, де ім'я користувача містить «&», і як цього уникнути? А з лапкою в адресі посилання?
5. Навіщо ділити довгий текст між рядками, а не кожні N символів?
6. Що робить розсилка, коли чат заблокував бота, і коли Telegram відповідає 429? Чому повторний збір нікого не сповіщає?
7. Чому webhook відповідає 200 одразу, а обробка — у фоні?

<details>
<summary>Відповіді</summary>

1. JSON з подією (повідомленням). У polling бот сам питає `getUpdates`; у webhook Telegram надсилає POST на наш https-адрес із секретним заголовком.
2. Middleware кладе значення в `data`, фільтр `Command` — `command`; aiogram передає handler ті ключі `data`, які збігаються з іменами його параметрів.
3. aiogram бере перший handler, чий фільтр підійшов, а `F.text` підходить до будь-якого тексту. Роутер можна підключити лише до одного диспетчера, а їх у процесі кілька.
4. Telegram відхилить усе повідомлення (`can't parse entities`), бот мовчатиме; рятує `esc(...)`. Лапка в `href` обриває атрибут — адресу екранують з `quote=True` (`link`).
5. Розріз посеред тегу чи сутності робить обидві частини невалідним HTML.
6. 403 — видаляє підписки чату й продовжує; 429 — чекає `retry_after` і пробує ще раз. `insert_new` повертає лише справді нові новини.
7. Telegram чекає відповіді недовго і, не дочекавшись, повторює update — бот відповів би кілька разів.
</details>

## Далі

- Книга курсу: [Урок 48](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_48/) — polling і webhook, сповіщення покроково, архітектура.
- У папці `news_hub`: `pytest tests/integration/test_bot.py -v`; бот наживо на двійнику — README, розділ «Telegram-бот».
- Уроки 49–51 — Docker, Compose, CI/CD.